# POC — Criação e Carga da Tabela `tb_ligacoes`
### Origem: `tb_dados_bruto` → Destino: `tb_ligacoes`
---
| Grupo           | Colunas                                                                 |
|-----------------|-------------------------------------------------------------------------|
| Identificação   | `codigo_ddd_to`                                                         |
| Datas e Horas   | `data_inicio`, `hora_inicio`, `data_fim`, `hora_fim`, `data_inicio_tabulacao`, `hora_inicio_tabulacao` |
| Dimensões (FK)  | `nm_dsc_campanha`, `nm_supervisor`, `id_dsc_contato`, `id_dsc_lista`, `id_dsc_mailing`, `id_dsc_status`, `id_dsc_tabulacao`, `id_dsc_usuario` |
|                 | `qt_tempo_tabulando` | 
| Indicadores     | 13 colunas `bl_*` (booleano)                                            |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import time

## 2. Configurações

In [ ]:
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_USER     = 'root'       # altere para seu usuário
DB_PASSWORD = 'senha'  # altere para sua senha
DB_NAME     = 'db_call_center'     # altere se necessário

TABLE_NAME  = 'tb_ligacoes'
CHUNK_SIZE  = 500          # linhas por lote (ajuste conforme volume)

## 3. Conexão com o banco

In [ ]:
connection_string = (
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

engine = create_engine(connection_string, echo=False)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Conexão OK!' AS status"))
    print(result.fetchone()[0])

## 4. DDL — Criação da tabela `tb_ligacoes`

- Sem índices secundários conforme especificação
- Booleanos armazenados como `TINYINT(1)` — padrão MySQL para `BOOLEAN`
- Datas e horas em colunas distintas para facilitar filtros no Power BI

In [ ]:
DDL = """
    CREATE TABLE IF NOT EXISTS tb_ligacoes (
        num_index               INT      NOT NULL AUTO_INCREMENT,   -- Chave primária
        id_lead                 INT      NULL,
        codigo_ddd_to           SMALLINT NULL,                      -- Identificação do telefone

        -- Datas e horas
        data_inicio              DATE    NULL,
        hora_inicio              TIME    NULL,
        qtd_segundos_hora_inicio INT     NULL,
        data_fim                 DATE    NULL,
        hora_fim                 TIME    NULL,
        qtd_segundos_hora_fim    INT     NULL,
        data_inicio_tabulacao    DATE    NULL,
        hora_inicio_tabulacao    TIME    NULL,
        qtd_segundos_inicio_tab  INT     NULL,
        qt_tempo_tabulando       INT     NULL, 

        -- Descritivos
        nm_dsc_campanha         VARCHAR(100) NULL,
        nm_supervisor           VARCHAR(100) NULL,

        -- Chaves estrangeiras (IDs das dimensões)
        id_dsc_contato          INT          NULL,
        id_dsc_lista            INT          NULL,
        id_dsc_mailing          INT          NULL,
        id_dsc_status           INT          NULL,
        id_dsc_tabulacao        INT          NULL,
        id_dsc_usuario          INT          NULL,

        -- Indicadores booleanos
        bl_atendido             TINYINT(1)   NULL,
        bl_abordagem            TINYINT(1)   NULL,
        bl_target               TINYINT(1)   NULL,
        bl_venda                TINYINT(1)   NULL,
        bl_agendamento          TINYINT(1)   NULL,
        bl_agendamento_pessoal  TINYINT(1)   NULL,
        bl_telefone_finalizado  TINYINT(1)   NULL,
        bl_finaliza_lead        TINYINT(1)   NULL,
        bl_pre_venda            TINYINT(1)   NULL,
        bl_venda                TINYINT(1)   NULL,
        bl_recusa               TINYINT(1)   NULL,
        bl_improdutivo          TINYINT(1)   NULL,
        bl_auditor              TINYINT(1)   NULL,
        bl_auditoria_backoffice TINYINT(1)   NULL,

        PRIMARY KEY (num_index)

    ) ENGINE=InnoDB
      DEFAULT CHARSET=utf8mb4
      COLLATE=utf8mb4_unicode_ci
      COMMENT='Tabela de ligações recebidas/efetuadas'
"""

with engine.connect() as conn:
    conn.execute(text(DDL))
    conn.commit()

print("Tabela tb_ligacoes criada (ou já existia)!")

## 5. Leitura da origem `tb_dados_bruto`

In [ ]:
print("📥 Lendo tb_dados_bruto...")

df_raw = pd.read_sql(
    "SELECT * FROM tb_dados_bruto",
    con=engine,
    dtype=str           # mantém tudo como string para transformar no Pandas
)

# Substitui strings vazias por NaN — facilita todo o tratamento posterior
df_raw = df_raw.replace('', np.nan)

print(f"Leitura concluída — {len(df_raw):,} linhas | {len(df_raw.columns)} colunas")

In [ ]:
df_raw.info()

## 6. Transformações

| Tipo        | Transformação Pandas                                         |
|-------------|--------------------------------------------------------------|
| DDD         | `str[:2]` → `pd.to_numeric` → `Int16`                       |
| Data        | `str[:10]` → `pd.to_datetime('%Y-%m-%d')` → `.dt.date`      |
| Hora        | `str[11:19]` → `pd.to_datetime('%H:%M:%S')` → `.dt.time`    |
| INT (id_*)  | `pd.to_numeric` → `Int64` (nullable)                        |
| BOOL (bl_*) | `pd.to_numeric` → `Int8` (nullable)                         |

In [ ]:
df = pd.DataFrame()

# ── DDD ────────────────────────────────────────────────────────
df['codigo_ddd_to'] = pd.to_numeric(
    df_raw['nu_dsc_telefone_to'].str[:2], errors='coerce'
).astype('Int16')

# ── Datas e Horas ──────────────────────────────────────────────
def parse_data(serie):
    return pd.to_datetime(serie.str[:10], format='%Y-%m-%d', errors='coerce').dt.date

def parse_hora(serie):
    return pd.to_datetime(serie.str[11:19], format='%H:%M:%S', errors='coerce').dt.time

def calc_segundos(serie):
    """Converte coluna datetime string em quantidade de segundos desde 00:00:00.
    Fórmula: (hora * 3600) + (minuto * 60) + segundo — espelha dim_horas.Qtd_Segundos.
    Retorna Int32 nullable (NaN para registros inválidos).
    """
    dt = pd.to_datetime(serie.str[11:19], format='%H:%M:%S', errors='coerce')
    segundos = (dt.dt.hour * 3600) + (dt.dt.minute * 60) + dt.dt.second
    return segundos.astype('Int32')

df['data_inicio']              = parse_data(df_raw['dh_inicio'])
df['hora_inicio']              = parse_hora(df_raw['dh_inicio'])
df['qtd_segundos_hora_inicio'] = calc_segundos(df_raw['dh_inicio'])

df['data_fim']                 = parse_data(df_raw['dh_fim'])
df['hora_fim']                 = parse_hora(df_raw['dh_fim'])
df['qtd_segundos_hora_fim']    = calc_segundos(df_raw['dh_fim'])

df['data_inicio_tabulacao']    = parse_data(df_raw['dh_inicio_tabulacao'])
df['hora_inicio_tabulacao']    = parse_hora(df_raw['dh_inicio_tabulacao'])
df['qtd_segundos_inicio_tab']  = calc_segundos(df_raw['dh_inicio_tabulacao'])

# ── Descritivos ────────────────────────────────────────────────
df['nm_dsc_campanha'] = df_raw['nm_dsc_campanha'].str.strip()
df['nm_supervisor']   = df_raw['nm_supervisor'].str.strip()

# ── IDs das dimensões (INT) ────────────────────────────────────
COLUNAS_INT = [
    'id_lead',
    'id_dsc_contato',
    'id_dsc_lista',
    'id_dsc_mailing',
    'id_dsc_status',
    'id_dsc_tabulacao',
    'id_dsc_usuario',
    'qt_tempo_tabulando'
]

for col in COLUNAS_INT:
    df[col] = pd.to_numeric(df_raw[col], errors='coerce').astype('Int64')

# ── Booleanos (TINYINT) ────────────────────────────────────────
COLUNAS_BOOL = [
    'bl_atendido', 
    'bl_abordagem', 
    'bl_target', 
    'bl_venda',
    'bl_agendamento', 
    'bl_agendamento_pessoal', 
    'bl_telefone_finalizado',
    'bl_finaliza_lead', 
    'bl_pre_venda', 
    'bl_recusa', 
    'bl_improdutivo',
    'bl_auditor', 
    'bl_auditoria_backoffice'
]

for col in COLUNAS_BOOL:
    df[col] = pd.to_numeric(df_raw[col], errors='coerce').astype('Int8')

print(f"Transformações aplicadas — {len(df):,} linhas antes dos filtros")

## 7. Filtro de valores negativos

Descarta qualquer linha onde **pelo menos uma** coluna numérica (`INT`, `BOOL` ou `DDD`)  
contenha valor `< 0`. Exibe um relatório por coluna com a quantidade de registros reprovados.

In [ ]:
total_antes = len(df)

COLUNAS_NUMERICAS = ['codigo_ddd_to'] + COLUNAS_INT + COLUNAS_BOOL

# Acumula máscara de linhas válidas (True = manter)
mascara_valida = pd.Series([True] * len(df), index=df.index)
reprovados_por_coluna = {}

for col in COLUNAS_NUMERICAS:
    negativos = df[col].notna() & (df[col] < 0)
    qtd = int(negativos.sum())
    if qtd > 0:
        reprovados_por_coluna[col] = qtd
    mascara_valida &= ~negativos

df = df[mascara_valida].reset_index(drop=True)

total_descartados = total_antes - len(df)

print(f"📊 Linhas antes do filtro  : {total_antes:,}")
print(f"📊 Linhas descartadas      : {total_descartados:,}")
print(f"📊 Linhas para carga       : {len(df):,}")

if reprovados_por_coluna:
    print("\n==>  Colunas com valores negativos detectados:")
    for col, qtd in reprovados_por_coluna.items():
        print(f"   {col:<30} → {qtd:,} registro(s) descartado(s)")
else:
    print("\nNenhum valor negativo encontrado — nenhum registro descartado.")

# ✅ Converte tipos nullable para tipos nativos Python (compatibilidade MySQL)
df = df.astype(object).where(df.notna(), other=None)

## 8. Carga por lotes na tabela `tb_ligacoes`

In [ ]:
total_rows    = len(df)
total_chunks  = (total_rows // CHUNK_SIZE) + (1 if total_rows % CHUNK_SIZE else 0)
rows_inserted = 0
start_time    = time.time()

print(f"🚀 Iniciando ingestão — {total_rows:,} linhas em {total_chunks} lote(s)...\n")

for i, chunk_start in enumerate(range(0, total_rows, CHUNK_SIZE), start=1):
    chunk = df.iloc[chunk_start : chunk_start + CHUNK_SIZE]

    chunk.to_sql(
        name      = TABLE_NAME,
        con       = engine,
        if_exists = 'append',  # nunca recria a tabela
        index     = False,     # num_index é AUTO_INCREMENT no banco
        method    = 'multi'    # insert em bloco (mais rápido)
    )

    rows_inserted += len(chunk)
    pct = (rows_inserted / total_rows) * 100
    print(f"   Lote {i:>3}/{total_chunks} — {rows_inserted:>6,}/{total_rows:,} linhas inseridas ({pct:.1f}%)")

elapsed = time.time() - start_time
print(f"\n Ingestão concluída em {elapsed:.2f}s — {rows_inserted:,} linhas gravadas em `{TABLE_NAME}`")

In [ ]:
total_rows    = len(df)
total_chunks  = (total_rows // CHUNK_SIZE) + (1 if total_rows % CHUNK_SIZE else 0)
rows_inserted = 0
start_time    = time.time()

print(f" Iniciando ingestão — {total_rows:,} linhas em {total_chunks} lote(s)...\n")

## 9. Validação pós-carga

In [ ]:
with engine.connect() as conn:
    count_destino = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
    count_origem  = conn.execute(text("SELECT COUNT(*) FROM tb_dados_bruto")).scalar()

esperado = count_origem - total_descartados

print(f" Linhas em tb_dados_bruto         : {count_origem:,}")
print(f" Linhas descartadas (negativos)    : {total_descartados:,}")
print(f" Linhas esperadas em {TABLE_NAME}  : {esperado:,}")
print(f" Linhas gravadas em {TABLE_NAME}   : {count_destino:,}")

if count_destino == esperado:
    print("\n Contagem OK — todos os registros válidos foram inseridos!")
else:
    print(f"\n* * *  Divergência de {abs(esperado - count_destino):,} linha(s) — verifique os logs.")

## 10. Preview dos dados no banco

In [ ]:
df_check = pd.read_sql(
    f"SELECT * FROM {TABLE_NAME} LIMIT 5",
    con=engine
)

print(f"Preview — primeiras 5 linhas de `{TABLE_NAME}`:")
df_check

## 11. Checagem de qualidade — nulos por coluna

In [ ]:
COLUNAS_CHECAR = [
    'id_lead', 'codigo_ddd_to', 
    'data_inicio',           'hora_inicio',           'qtd_segundos_hora_inicio',
    'data_fim',              'hora_fim',              'qtd_segundos_hora_fim',
    'data_inicio_tabulacao', 'hora_inicio_tabulacao', 'qtd_segundos_inicio_tab', 'qt_tempo_tabulando'
    'nm_dsc_campanha', 'nm_supervisor',
    'id_dsc_contato', 'id_dsc_lista',     'id_dsc_mailing',
    'id_dsc_status',  'id_dsc_tabulacao', 'id_dsc_usuario',
    'bl_atendido',    'bl_abordagem',           'bl_target',              'bl_venda',
    'bl_agendamento', 'bl_agendamento_pessoal', 'bl_telefone_finalizado', 'bl_finaliza_lead', 
    'bl_pre_venda',   'bl_recusa',              'bl_improdutivo',         'bl_auditor',
    'bl_auditoria_backoffice'
]

null_checks = ", ".join(
    [f"SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {col}" for col in COLUNAS_CHECAR]
)

df_nulls = pd.read_sql(f"SELECT {null_checks} FROM {TABLE_NAME}", con=engine)
df_nulls = df_nulls.T.rename(columns={0: 'qtd_nulos'})
df_nulls.index.name = 'coluna'
df_nulls['pct_nulos'] = (df_nulls['qtd_nulos'] / count_destino * 100).round(2).astype(str) + '%'

print(f"Checagem de nulos em `{TABLE_NAME}` ({count_destino:,} linhas total):")
df_nulls